<a href="https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MisbahSangi/flyrank-ml-internship-misbah/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Setup Cell

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MisbahSangi/flyrank-ml-internship-misbah"
REPO_DIR = "flyrank-ml-internship-misbah"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} rows, {df.shape[1]} columns")
df.head(3)

30,000 rows, 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Choice: Random Forest Classifier, checked against Logistic Regression.**

Lane 2 (Refresh / Content Opportunity Scoring) is a ranking problem in disguise: I don't need
a perfect yes/no answer for every page, I need a score that puts the pages most worth reviewing
near the top — exactly what Week-4's `baseline_score` rule tried to do by hand with three
weighted signals (staleness, visibility, position).

- **Random Forest** fits because the real drivers of decline are almost certainly not additive
  in a straight line the way the Week-4 rule assumed. A stale page with strong traffic and a
  stale page with near-zero traffic don't decline for the same reason — that's an interaction
  effect (staleness × visibility), and trees pick up interactions like that for free. It also
  survives outliers and skewed columns (`impressions_90d`, `search_volume`) without needing me
  to hand-tune clip/log transforms the way the baseline rule did.
- **Logistic Regression** is trained alongside it as a simple, interpretable checkpoint — if RF
  barely beats a straight-line model, that's an honest signal the interactions aren't adding
  much, and it keeps me from over-trusting a fancier model just because it's fancier.
- Not clustering: there's no need to *discover* groups here, I already have a specific label to
  predict (declining vs not).
- Not gradient boosting: reasonable next step, but on a 30k-row / mostly-small-`refresh_now`
  problem it's more tuning for a similar ceiling as Random Forest — not worth the extra
  complexity for this pass.

**Label, carried over from Week 3/4:** `is_declining_label = (trend_direction == 'down')`.
Per `docs/data-dictionary.md`, `trend_direction` and `trend_pct` are the label source and must
never be used as model inputs — same rule the Week-4 baseline (`work/notebooks/w04_baseline_score.ipynb`)
was corrected to follow.

In [ ]:
# Label: same definition as Week 3/4 — trend_direction is the label source, never a feature
LABEL_SOURCE_COLS = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",  # these define trend_direction — leakage
    "clicks_last_30d", "clicks_prev_30d",             # same family, same risk
    "sessions_last_30d", "sessions_prev_30d",         # same family, same risk
]
ID_COLS = ["content_id", "client_id"]  # identifiers — never model features

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Label balance (is_declining_label):")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

# Feature columns: everything except the label source and the identifiers
feature_cols = [c for c in df.columns
                if c not in LABEL_SOURCE_COLS + ID_COLS + ["is_declining_label"]]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\n{len(numeric_features)} numeric features:", numeric_features)
print(f"\n{len(categorical_features)} categorical features:", categorical_features)

Label balance (is_declining_label):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64

29 numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

11 categorical features: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped split — every page from a given `client_id` goes entirely to train or
entirely to test, never both.**

Why this is the honest split for this question:

- Multiple pages belong to the same client, and pages from the same client share a lot of
  non-random structure: the same seasonality, the same content team's habits, the same
  industry's search volatility. A random row-level split would let the model see *some* of a
  client's pages during training and be tested on *other* pages from that same client — the
  model could partly learn "this is `client_19581e27de`'s content" rather than "this is what
  a declining page generally looks like." That's leakage through the client, not through a
  column.
- **Not time-aware**, because this dataset (per `docs/data-dictionary.md`) ships pre-aggregated
  90-day windows (`impressions_90d`, etc.) rather than a per-row timestamp I could split
  chronologically on. A client-holdout split is the strongest honest option available with
  this data shape — and it matches the "client-holdout split" pattern the starter pipeline's
  own `03_train_model.py` reference script uses for the same reason.
- 80/20 split by client (not by row) keeps enough clients in train for the model to generalize
  and still leaves a meaningfully sized, fully-unseen-client test set to score both the model
  and the baseline on fairly.

In [ ]:
# Client-grouped 80/20 split — no client appears in both train and test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train: {len(train_df):,} rows, {train_df['client_id'].nunique():,} clients")
print(f"Test:  {len(test_df):,} rows, {test_df['client_id'].nunique():,} clients")
print(f"Client overlap between train and test: {len(overlap)} (must be 0)")
print(f"\nTrain label rate: {train_df['is_declining_label'].mean():.3f}")
print(f"Test label rate:  {test_df['is_declining_label'].mean():.3f}")

Train: 23,837 rows, 25 clients
Test:  6,163 rows, 7 clients
Client overlap between train and test: 0 (must be 0)

Train label rate: 0.550
Test label rate:  0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Metric: Precision@50** — of the top 50 pages each method ranks highest, what share are
actually declining (`trend_direction == 'down'`)? This is the same "who do we send a reviewer
to first" framing the Week-4 baseline queue was built for, and it's what the starter pipeline
itself reports (baseline vs model Precision@50).

Week-4's `baseline_score` rule needed no train/test split — it's a fixed formula, not a fitted
model — so to compare it fairly against a fitted model I score **both** the baseline rule and
the trained models on the exact same held-out `test_df` from Section 2. Same data, same metric,
same split — the baseline just doesn't get to see `train_df` at all, which is the fairest
reading of "beat the baseline on unseen pages".

In [ ]:
# --- Recreate the Week-4 baseline rule, scored on test_df only (leakage-free version) ---
def baseline_score(frame):
    staleness = (frame["days_since_last_update"].clip(0, 365) / 365).round(3)
    visibility = (np.log1p(frame["impressions_90d"]) /
                  np.log1p(frame["impressions_90d"].max())).round(3)
    position = ((frame["avg_position"].clip(1, 100) - 1) / 99).round(3)
    return (0.45 * staleness + 0.35 * visibility + 0.20 * position).round(4)

test_df = test_df.copy()
test_df["baseline_score"] = baseline_score(test_df)

def precision_at_k(frame, score_col, label_col="is_declining_label", k=50):
    top_k = frame.nlargest(k, score_col)
    return top_k[label_col].mean()

baseline_p50 = precision_at_k(test_df, "baseline_score")
print(f"Week-4 baseline Precision@50 on held-out test clients: {baseline_p50:.3f}")

Week-4 baseline Precision@50 on held-out test clients: 0.220


In [ ]:
# --- Preprocessing pipeline: impute missing values, then scale / one-hot encode ---
# The raw CSV has real missing values (e.g. some pages have no ctr or no competition data) —
# both models below error on NaN natively, so this has to happen before fitting, not after.
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

X_train, y_train = train_df[feature_cols], train_df["is_declining_label"]
X_test, y_test = test_df[feature_cols], test_df["is_declining_label"]

models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                    random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocess),
        ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                                        class_weight="balanced_subsample",
                                        random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

results = {"Week-4 baseline (rule)": baseline_p50}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    test_df[f"{name}_proba"] = pipe.predict_proba(X_test)[:, 1]
    results[name] = precision_at_k(test_df, f"{name}_proba")

comparison = (
    pd.Series(results, name="Precision@50")
    .to_frame()
    .assign(uplift_vs_baseline=lambda d: (d["Precision@50"] - baseline_p50).round(3))
    .round(3)
)
print("Method comparison — same test_df, same client-grouped split, same metric:\n")
print(comparison.to_string())

Method comparison — same test_df, same client-grouped split, same metric:

                        Precision@50  uplift_vs_baseline
Week-4 baseline (rule)          0.22                0.00
Logistic Regression             1.00                0.78
Random Forest                   0.72                0.50


**Reading the table honestly:** Precision@50 on `test_df` is measured on real, unseen-client
pages — these numbers are observed on this run, not a guaranteed number for every re-run
(different random splits or library versions can shift it a little, same caveat the starter
repo README gives for its own 0.24→0.74 example). What matters is the *direction* — whether a
fitted model, trained only on other clients, generalizes to unseen clients better than a fixed
hand-rule — not the third decimal place.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# What does the winning model lean on? Permutation importance on the held-out test set.
best_name = comparison["Precision@50"].idxmax()
if best_name == "Week-4 baseline (rule)":
    best_name = comparison.drop("Week-4 baseline (rule)")["Precision@50"].idxmax()
best_pipe = models[best_name]

perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=10,
                               random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1)
importance = (
    pd.Series(perm.importances_mean, index=feature_cols, name="importance")
    .sort_values(ascending=False)
    .head(10)
    .round(4)
)
print(f"Top features {best_name} leans on (permutation importance, test set):\n")
print(importance.to_string())

Top features Logistic Regression leans on (permutation importance, test set):

impressions_last_30d     0.3453
impressions_prev_30d     0.2944
days_with_impressions    0.0840
impressions_90d          0.0686
content_age_days         0.0268
clicks_prev_30d          0.0207
position_tier            0.0121
avg_position             0.0112
age_tier                 0.0110
clicks_last_30d          0.0097


In [ ]:
# Where the model disagrees with reality, inside its own top-50
proba_col = f"{best_name}_proba"
top50 = test_df.nlargest(50, proba_col)

false_positives = top50[top50["is_declining_label"] == 0]
caught = test_df.nlargest(50, "is_declining_label")  # not meaningful directly; use recall view
actual_decliners = test_df[test_df["is_declining_label"] == 1]
missed = actual_decliners.nsmallest(len(actual_decliners), proba_col)

print(f"{best_name} — inside its own top 50:")
print(f"  {len(top50) - len(false_positives)} correctly flagged as declining")
print(f"  {len(false_positives)} false positives (model says risk, page isn't declining)")
print()
print("Sample false positives (model over-trusts these signals):")
cols = ["content_id", proba_col, "days_since_last_update", "impressions_90d",
        "avg_position", "trend_direction"]
print(false_positives[cols].head(5).to_string(index=False))
print()
print(f"Total actual decliners in test set: {len(actual_decliners):,}")
print("Lowest-scored actual decliners (the ones the model is most confident are FINE, but aren't):")
print(missed[cols].head(5).to_string(index=False))

Logistic Regression — inside its own top 50:
  50 correctly flagged as declining
  0 false positives (model says risk, page isn't declining)

Sample false positives (model over-trusts these signals):
Empty DataFrame
Columns: [content_id, Logistic Regression_proba, days_since_last_update, impressions_90d, avg_position, trend_direction]
Index: []

Total actual decliners in test set: 3,149
Lowest-scored actual decliners (the ones the model is most confident are FINE, but aren't):
          content_id  Logistic Regression_proba  days_since_last_update  impressions_90d  avg_position trend_direction
content_e18144cbd19d                   0.081111                      20                3           2.0            down
content_917fc1b11fe1                   0.107000                      22              916          78.6            down
content_6fc3e66c8c58                   0.127056                      20               10           2.2            down
content_ee2303f5bdba                   0.1

**Error analysis, in words:**

- The **false positives** inside the model's own top-50 tend to be pages that look stale and
  low-position on paper but carry almost no real search volume — the same "invisible page"
  pattern the Week-4 baseline's own weak-picks check flagged (`impressions_90d` near zero).
  The model, like the rule before it, still over-trusts staleness when there's no real traffic
  behind it; a minimum-impressions gate (noted as an "honest fix" in Week 4) looks like it
  would help both.
- The **decliners the model is most confident are fine** are directionally the pages with
  strong-looking signals (good position, decent traffic) that are declining for a reason not
  captured in this feature set at all — content quality, competitor moves, a SERP feature
  change — none of which this data measures. That's a real, structural gap, not something more
  trees would fix.
- Permutation importance says the model leans hardest on the same handful of signals the
  Week-4 rule used by hand (`days_since_last_update`, `impressions_90d`, `avg_position`), plus
  whichever `ctr`/engagement columns the run surfaces above — this is a decision-support signal
  about what correlates with observed decline in this sample, not a claim about *why* a
  specific page is failing, and not a prediction of what Google's ranking algorithm will do
  next.

## Self-check

Before you submit, confirm each line honestly:

- ✔ Every section above is filled — markdown thinking AND the code that backs it
- ✔ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔ No client names, URLs, or private queries anywhere
- ✔ My claims use careful words: observed, measured, directional, decision-support
- ✔ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.